In [2]:
# tool calling in ollama

import ollama

In [4]:
# database
inventory_db = {
    "laptop" : {"stock" : 2, "base_price" : 12000},
    "iphone" : {"stock" : 3, "base_price" : 24000},
    "monitor" : {"stock" : 0, "base_price" : 6000}
}
print(inventory_db)

{'laptop': {'stock': 2, 'base_price': 12000}, 'iphone': {'stock': 3, 'base_price': 24000}, 'monitor': {'stock': 0, 'base_price': 6000}}


In [10]:
# define tools 

# tool 1 :
def check_inventory(product_name):
    product_name = product_name.lower()

    if product_name in inventory_db:
        return inventory_db[product_name]
    
    return {'stock': 0, 'base_price': None}

# tool 2 :
def calculate_loyalty_discount(base_price, year_as_customer):
    discount = min(year_as_customer * 0.05, 0.30) # max discount is 30%

    discount_price = base_price * (1 - discount)

    return round(discount_price,2)


In [8]:
# define schema

tools = [
    {
        "type" : "function",
        "function" : {
            "name" : "check_inventory",
            "description" : "check stock and price of the product.",
            "parameters" : {"type" : "object" , "properties" :{"product_name" : {"type" : "string"}}},
            "required" : ["product_name"]
        }
    },
    {
        "type" : "function",
        "function" : {
            "name" : "calculate_loyalty_discount",
            "description" : "calculate the discount for customers based on loyalty years.",
            "parameters" : {"type" : "object" , "properties" :{"base_price" : {"type" : "number"},"year_as_customer" : {"type" : "integer"}}},
            "required" : ["base_price","year_as_customer"]
        }
    }
]


In [9]:
# I want to buy an iphone. can you check the stock?

message = [
    {"role" : "user","content" : "I want to buy a iphone. can you check the stock?"}
]

# calling LLMs
response = ollama.chat(
    model = "qwen3-vl:2b",
    messages=message,
    tools=tools
)

print(response["message"])

role='assistant' content='' thinking='Okay, let me see. The user wants to buy an iPhone and asks if I can check the stock. First, I need to figure out what tools are available.\n\nLooking at the provided functions: there\'s check_inventory which takes product_name. It says it checks stock and price for a product. But wait, the user mentioned "iPhone", so maybe they want to check if there\'s stock for iPhone models.\n\nWait, but the function parameters require product_name as a string. So the correct approach is to call check_inventory with "iPhone" as the argument. However, the problem here might be that the user specifically says "I want to buy a iphone." Maybe it\'s a typo for "iPhone". Also, in some contexts, maybe they\'re referring to a specific model. But since the function only takes product_name, I should assume it\'s the main product name.\n\nWait, the user is asking about stock of iPhone. The check_inventory function checks stock and price. But there\'s also another tool: cal

In [13]:
# map the function so the script can call them by name

available_functions = {
    "check_inventory" : check_inventory,
    "calculate_loyalty_discount" : calculate_loyalty_discount
}

In [14]:
# Check if the model asked to use any tools
tool_calls = response['message'].get('tool_calls')

if tool_calls:
    for tool_call in tool_calls:
        tool_name = tool_call['function']['name']
        tool_args = tool_call['function']['arguments']

        # Get the actual Python function
        function_to_call = available_functions[tool_name]

        # Run the function with given arguments
        result = function_to_call(**tool_args)

        # Add the model's tool request to the conversation
        message.append(response['message'])

        # Add the tool's result back to the conversation
        message.append({
            'role': 'tool',
            'content': str(result)
        })

message

[{'role': 'user',
  'content': 'I want to buy a iphone. can you check the stock?'},
 Message(role='assistant', content='', thinking='Okay, let me see. The user wants to buy an iPhone and asks if I can check the stock. First, I need to figure out what tools are available.\n\nLooking at the provided functions: there\'s check_inventory which takes product_name. It says it checks stock and price for a product. But wait, the user mentioned "iPhone", so maybe they want to check if there\'s stock for iPhone models.\n\nWait, but the function parameters require product_name as a string. So the correct approach is to call check_inventory with "iPhone" as the argument. However, the problem here might be that the user specifically says "I want to buy a iphone." Maybe it\'s a typo for "iPhone". Also, in some contexts, maybe they\'re referring to a specific model. But since the function only takes product_name, I should assume it\'s the main product name.\n\nWait, the user is asking about stock of i

In [16]:
# Ask the model to generate the final answer
final_response = ollama.chat(
    model='qwen3-vl:2b',
    messages=message
)

print(final_response['message']['content'])

Here's the current inventory status for your iPhone (based on the system check):

- **Stock Available**: 3 units  
- **Base Price**: ¥24,000  

This indicates there are currently 3 iPhones in stock with a standard price of ¥24,000. Would you like to explore more details about availability for specific models (e.g., iPhone 15/14) or check other product variations?
